## Packages

In [ ]:
import spatialdata as spd
import matplotlib.pyplot as plt
import anndata as ad
import squidpy as sq
import numpy as np
import spatialdata as spd

In [ ]:
from pathlib import Path

# Data

In [ ]:
# proj_folder   = Path("/coh_labs/yunroseli/Jona/CAR-T/") # Most up to date files
# zarr_file     = proj_folder / "data/zarr/CellCharterClusters_c2l_annotated"
zarr_file     = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400"
# code_folder   = Path("/home/janzules/spatial/CAR-T/code")
# hallmark_file = code_folder / "references/Mouse_Hallmark.gmt"

## Loading Data

In [ ]:
#Reading in files

sdata = spd.read_zarr(zarr_file)
adata = sdata.tables['segmentation_counts']

In [ ]:
# Dropping unknown cells
adata = adata[
    (adata.obs['c2l_permissive'] != 'Unknown') & 
    (adata.obs['c2l_permissive'] != 'Erythrocyte')
].copy()


In [ ]:
adata.obs.columns

# Visualizing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# ----------------------------
# User settings
# ----------------------------
cell_col = "c2l_permissive"
sample_col = "mouse"       # most specific tissue/sample label
condition_col = "treatment"  # or "condition", depending on your obs columns

# ----------------------------
# Basic checks
# ----------------------------
required_cols = [cell_col, sample_col, condition_col]
missing = [c for c in required_cols if c not in adata.obs.columns]

if missing:
    raise ValueError(f"Missing columns in adata.obs: {missing}")

obs = adata.obs[required_cols].copy()
obs = obs.dropna(subset=[cell_col, sample_col, condition_col])

obs[cell_col] = obs[cell_col].astype(str)
obs[sample_col] = obs[sample_col].astype(str)
obs[condition_col] = obs[condition_col].astype(str)

# ----------------------------
# Counts and fractions per sample
# ----------------------------
sample_counts = pd.crosstab(obs[sample_col], obs[cell_col])
sample_fractions = sample_counts.div(sample_counts.sum(axis=1), axis=0)

sample_metadata = (
    obs[[sample_col, condition_col]]
    .drop_duplicates()
    .set_index(sample_col)
)

sample_fractions = sample_fractions.join(sample_metadata)

print("Sample-level count table:")
display(sample_counts.head())

print("Sample-level fraction table:")
display(sample_fractions.head())

print("\nNumber of cells per sample:")
display(sample_counts.sum(axis=1).sort_values(ascending=False))

In [ ]:
# Separate numeric cell-type columns from metadata
cell_type_cols = sample_counts.columns.tolist()

# Optional: sort samples by condition, then sample name
plot_df = sample_fractions.copy()
plot_df = plot_df.sort_values([condition_col, sample_col])

plot_cell_frac = plot_df[cell_type_cols]

fig, ax = plt.subplots(figsize=(14, 6))

bottom = np.zeros(plot_cell_frac.shape[0])

for ct in cell_type_cols:
    ax.bar(
        plot_cell_frac.index,
        plot_cell_frac[ct],
        bottom=bottom,
        label=ct
    )
    bottom += plot_cell_frac[ct].values

ax.set_ylabel("Fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Sample")
ax.set_title("Cellular composition per sample")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

print("Summary:")
print("Each bar is one tissue/sample. Fractions are normalized within sample, so all bars sum to 100%.")

In [ ]:
# Define rare cell types based on global abundance
# Rare cell types lumped to 1%
min_global_fraction = 0.01  # 1%

global_fraction = obs[cell_col].value_counts(normalize=True)
major_cell_types = global_fraction[global_fraction >= min_global_fraction].index.tolist()
rare_cell_types = global_fraction[global_fraction < min_global_fraction].index.tolist()

print("Major cell types:")
print(major_cell_types)

print("\nCollapsed into Other:")
print(rare_cell_types)

obs_collapsed = obs.copy()
obs_collapsed["cell_type_collapsed"] = np.where(
    obs_collapsed[cell_col].isin(major_cell_types),
    obs_collapsed[cell_col],
    "Other"
)

collapsed_counts = pd.crosstab(obs_collapsed[sample_col], obs_collapsed["cell_type_collapsed"])
collapsed_fractions = collapsed_counts.div(collapsed_counts.sum(axis=1), axis=0)

collapsed_fractions = collapsed_fractions.join(sample_metadata)
collapsed_fractions = collapsed_fractions.sort_values([condition_col, sample_col])

collapsed_cell_cols = collapsed_counts.columns.tolist()
plot_df = collapsed_fractions[collapsed_cell_cols]

fig, ax = plt.subplots(figsize=(12, 5))

bottom = np.zeros(plot_df.shape[0])

for ct in collapsed_cell_cols:
    ax.bar(
        plot_df.index,
        plot_df[ct],
        bottom=bottom,
        label=ct
    )
    bottom += plot_df[ct].values

ax.set_ylabel("Fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Sample")
ax.set_title("Cellular composition per sample, rare cell types collapsed")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

print("Summary:")
print(f"Cell types below {min_global_fraction:.0%} global abundance were collapsed into Other.")

# Lineage-level composition

In [ ]:
lineage_map = {
    "Cancer_cell": "Tumor",

    "N1_like_Neu": "Neutrophil",
    "N2_like_Neu": "Neutrophil",

    "M1_like_Mac": "Macrophage/Monocyte",
    "M2_like_Mac": "Macrophage/Monocyte",
    "Intermediate_Mac": "Macrophage/Monocyte",
    "Classical_Mono": "Macrophage/Monocyte",
    "Nonclassical_Mono": "Macrophage/Monocyte",

    "CD8_T": "Lymphoid",
    "CD4_T": "Lymphoid",
    "Treg": "Lymphoid",
    "NK": "Lymphoid",
    "NKT": "Lymphoid",
    "B": "Lymphoid",

    "cDC": "Dendritic cell",
    "pDC": "Dendritic cell",

    "Fibroblast": "Stromal",
    "Endothelial": "Stromal",

    "Erythrocyte": "Erythrocyte"
}

obs_lineage = obs.copy()
obs_lineage["lineage"] = obs_lineage[cell_col].map(lineage_map).fillna("Other")

lineage_counts = pd.crosstab(obs_lineage[sample_col], obs_lineage["lineage"])
lineage_fractions = lineage_counts.div(lineage_counts.sum(axis=1), axis=0)

lineage_fractions = lineage_fractions.join(sample_metadata)
lineage_fractions = lineage_fractions.sort_values([condition_col, sample_col])

lineage_cols = lineage_counts.columns.tolist()
plot_df = lineage_fractions[lineage_cols]

fig, ax = plt.subplots(figsize=(12, 5))

bottom = np.zeros(plot_df.shape[0])

for lineage in lineage_cols:
    ax.bar(
        plot_df.index,
        plot_df[lineage],
        bottom=bottom,
        label=lineage
    )
    bottom += plot_df[lineage].values

ax.set_ylabel("Fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Sample")
ax.set_title("Lineage-level cellular composition per sample")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Lineage"
)

plt.tight_layout()
plt.show()

print("Summary:")
print("This plot collapses detailed cell annotations into broader biological compartments.")

In [ ]:
# Statistical measure per group condition

condition_summary = (
    sample_fractions
    .groupby(condition_col)[cell_type_cols]
    .agg(["mean", "std", "sem"])
)

display(condition_summary)

condition_mean = sample_fractions.groupby(condition_col)[cell_type_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))

bottom = np.zeros(condition_mean.shape[0])

for ct in cell_type_cols:
    ax.bar(
        condition_mean.index,
        condition_mean[ct],
        bottom=bottom,
        label=ct
    )
    bottom += condition_mean[ct].values

ax.set_ylabel("Mean fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Treatment group")
ax.set_title("Mean cellular composition by treatment group")
ax.tick_params(axis="x", rotation=45)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

print("Summary:")
print("Treatment-group bars are calculated from per-sample fractions, not pooled cells.")

In [ ]:
cell_of_interest = "CD8_T"

focus_df = sample_fractions[[cell_of_interest, condition_col]].copy()
focus_df = focus_df.rename(columns={cell_of_interest: "fraction"})
focus_df = focus_df.sort_values(condition_col)

fig, ax = plt.subplots(figsize=(7, 5))

conditions = focus_df[condition_col].unique()
x_positions = np.arange(len(conditions))

for i, cond in enumerate(conditions):
    y = focus_df.loc[focus_df[condition_col] == cond, "fraction"]

    # sample dots
    ax.scatter(
        np.repeat(i, len(y)),
        y,
        alpha=0.8
    )

    # condition mean
    ax.hlines(
        y.mean(),
        i - 0.25,
        i + 0.25,
        linewidth=3
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(conditions, rotation=45, ha="right")
ax.set_ylabel(f"{cell_of_interest} / all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"{cell_of_interest} abundance by treatment group")

plt.tight_layout()
plt.show()

print("Summary:")
print(f"This plot shows {cell_of_interest} as a fraction of all segmented cells in each sample.")

In [ ]:
immune_cell_types = [
    "N2_like_Neu",
    "M2_like_Mac",
    "Classical_Mono",
    "N1_like_Neu",
    "NK",
    "CD8_T",
    "M1_like_Mac",
    "B",
    "cDC",
    "CD4_T",
    "Intermediate_Mac",
    "Nonclassical_Mono",
    "pDC",
    "NKT",
    "Treg"
]

cell_of_interest = "CD8_T"

immune_obs = obs[obs[cell_col].isin(immune_cell_types)].copy()

immune_counts = pd.crosstab(immune_obs[sample_col], immune_obs[cell_col])
immune_fractions = immune_counts.div(immune_counts.sum(axis=1), axis=0)

immune_fractions = immune_fractions.join(sample_metadata)

focus_df = immune_fractions[[cell_of_interest, condition_col]].copy()
focus_df = focus_df.rename(columns={cell_of_interest: "fraction"})
focus_df = focus_df.dropna()
focus_df = focus_df.sort_values(condition_col)

fig, ax = plt.subplots(figsize=(7, 5))

conditions = focus_df[condition_col].unique()
x_positions = np.arange(len(conditions))

for i, cond in enumerate(conditions):
    y = focus_df.loc[focus_df[condition_col] == cond, "fraction"]

    ax.scatter(
        np.repeat(i, len(y)),
        y,
        alpha=0.8
    )

    ax.hlines(
        y.mean(),
        i - 0.25,
        i + 0.25,
        linewidth=3
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(conditions, rotation=45, ha="right")
ax.set_ylabel(f"{cell_of_interest} / immune cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"{cell_of_interest} fraction among immune cells")

plt.tight_layout()
plt.show()

print("Summary:")
print(f"This plot shows {cell_of_interest} as a fraction of immune cells only.")